In [ ]:
# ── Setup (run this cell first) ──────────────────────────────────────────────
import numpy as np
import json

matrix = np.load('matrices/mumbai_matrix.npy')
meta   = json.load(open('matrices/mumbai_meta.json'))
names  = meta['place_names']

print(f"Shape  : {matrix.shape}")
print(f"Places : {meta['total_places']}")
reachable = matrix[matrix > 0]
print(f"Min dist (reachable) : {reachable.min()/1000:.2f} km")
print(f"Max dist             : {reachable.max()/1000:.2f} km")
print(f"Avg dist             : {reachable.mean()/1000:.2f} km")
print(f"No-route pairs (-1)  : {int((matrix == -1).sum())}")

In [ ]:
# ── Styled distance table (8×8 corner) ───────────────────────────────────────
import numpy as np, json, pandas as pd

matrix = np.load('matrices/mumbai_matrix.npy')
meta   = json.load(open('matrices/mumbai_meta.json'))
names  = meta['place_names']

N = 8
labels = [n[:22] for n in names[:N]]
block  = matrix[:N, :N] / 1000
block[block < 0] = float('nan')  # -1 → NaN so it shows blank

df = pd.DataFrame(block, index=labels, columns=labels)
df.style.format("{:.1f} km", na_rep="—").background_gradient(cmap='YlOrRd', axis=None)

In [ ]:
# ── Look up distance between two places ──────────────────────────────────────
import numpy as np, json

matrix = np.load('matrices/mumbai_matrix.npy')
meta   = json.load(open('matrices/mumbai_meta.json'))
names  = meta['place_names']

def get_distance(name_a: str, name_b: str):
    def find(q):
        for i, n in enumerate(names):
            if q.lower() in n.lower():
                return i, n
        return None, None
    i, a = find(name_a)
    j, b = find(name_b)
    if i is None: print(f"'{name_a}' not found"); return
    if j is None: print(f"'{name_b}' not found"); return
    fmt = lambda v: f"{v/1000:.2f} km" if v >= 0 else "no route"
    print(f"{a}  →  {b} : {fmt(matrix[i][j])}")
    print(f"{b}  →  {a} : {fmt(matrix[j][i])}")

# ← change these to any place names
get_distance('Gateway', 'Juhu')
get_distance('Bandra', 'Marine')

In [ ]:
# ── Find nearest N places to a given place ────────────────────────────────────
import numpy as np, json

matrix = np.load('matrices/mumbai_matrix.npy')
meta   = json.load(open('matrices/mumbai_meta.json'))
names  = meta['place_names']

def nearest(name: str, top_n: int = 5):
    idx = next((i for i, n in enumerate(names) if name.lower() in n.lower()), None)
    if idx is None:
        print(f"'{name}' not found"); return
    row = matrix[idx].copy()
    row[idx]   = float('inf')
    row[row < 0] = float('inf')
    top = np.argsort(row)[:top_n]
    print(f"Nearest {top_n} to '{names[idx]}':")
    for rank, j in enumerate(top, 1):
        print(f"  {rank}. {names[j][:45]:45}  {row[j]/1000:.2f} km")

# ← change to any place name
nearest('Gateway', top_n=5)

In [ ]:
# ── Print full place list with indices ───────────────────────────────────────
import json

meta  = json.load(open('matrices/mumbai_meta.json'))
names = meta['place_names']
cats  = meta['categories']

print(f"{'Idx':>3}  {'Category':<22}  Place")
print("-" * 70)
for i, (n, c) in enumerate(zip(names, cats)):
    print(f"{i:>3}  {c:<22}  {n}")